In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "5. comparison_models_loo"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import sys

# Make the adjacent src/ package importable when this notebook is run from the notebooks folder.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import src.utils_val as uv

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import ast
from scipy.stats import pearsonr
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, r2_score
from collections import Counter

# Setting plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Inter', 'Roboto', 'Arial']

OUTPUT_DIR = "../results_comparison_all_models"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


In [ ]:
VAL_DATA_DIR = "../val_data"

# Model Result Directories
MODEL_DIRS = {
    'Model_Z':    "../results_val",
    'Model_Rank': "../results_rk",
    'Model_Bin':  "../results_bin",
    'Model_None': "../results_none"
}


In [ ]:
model_dfs = {}
all_proteins = []

for m_name, m_dir in MODEL_DIRS.items():
    print(f"Loading {m_name} from {m_dir}...")
    # reads <P>_best.parquet (pipeline default) or <P>_best.csv
    full_df, best_files = uv.load_best_result_tables(m_dir)
    dfs = [full_df] if not full_df.empty else []

    if dfs:
        full_df = full_df.drop_duplicates('protein', keep='last')
        # Create dictionary for quick lookup: protein -> coeff_dict
        model_dfs[m_name] = full_df.set_index('protein')['feat_coeffs_dict'].map(uv.parse_coefficients).to_dict()
        print(f"  Loaded {len(model_dfs[m_name])} models.")
        all_proteins.append(set(model_dfs[m_name].keys()))
    else:
        print(f"  No models found for {m_name}")
        model_dfs[m_name] = {}
        all_proteins.append(set())

# Find Intersection
if all_proteins:
    common_proteins = set.intersection(*all_proteins)
    common_proteins = sorted(list(common_proteins))
    print(f"\nIntersection: {len(common_proteins)} proteins common to all 4 models.")
else:
    common_proteins = []
    print("No models loaded.")


In [ ]:
datasets = ['iTRAQ', 'LabelFree', 'TMT']
comparison_results = []

for ds in datasets:
    RNA_PATH = os.path.join(VAL_DATA_DIR, f"{ds}_transcriptomics.joblib")
    PROT_PATH = os.path.join(VAL_DATA_DIR, f"{ds}_proteomics.joblib")
    
    if not (os.path.exists(RNA_PATH) and os.path.exists(PROT_PATH)):
        continue
        
    print(f"\nEvaluating on {ds} dataset...")
    rna_df = joblib.load(RNA_PATH).fillna(0)
    prot_df = joblib.load(PROT_PATH)
    
    available_feats = set(rna_df.columns)
    
    # Store results for this dataset
    ds_res = {'Dataset': ds}
    
    # Evaluate each model type ON THE COMMON PROTEINS ONLY
    for m_name in MODEL_DIRS.keys():
        y_preds = []
        y_trues = []
        
        warned = False
        
        for p in common_proteins:
            if p not in prot_df.columns: 
                continue 
            
            # Get model coefficients
            coef_str = model_dfs[m_name][p]
            coeffs = uv.parse_coefficients(coef_str)
            needed = list(coeffs.keys())
            
            # FIX: Check for missing features and impute 0
            current_missing = [f for f in needed if f not in available_feats]
            
            if current_missing:
                if not warned:
                    print(f"  [{m_name}] Note: Missing features for {p} (and others). Imputing 0.")
                    warned = True
                
                present_feats = [f for f in needed if f in available_feats]
                if not present_feats:
                     yp = np.zeros(rna_df.shape[0])
                else:
                     weights_present = np.array([coeffs[f] for f in present_feats])
                     X_present = rna_df[present_feats]
                     yp = X_present.dot(weights_present)
            else:
                # Standard path
                weights = np.array([coeffs[f] for f in needed])
                X = rna_df[needed]
                yp = X.dot(weights)

            yt = prot_df[p]
            
            y_preds.append(yp.values)
            y_trues.append(yt.values)
            
        # Convert to matrix
        if not y_preds:
            print(f"  No valid predictions for {m_name} on {ds}")
            continue
            
        yp_mat = np.array(y_preds).T
        yt_mat = np.array(y_trues).T
        
        # Calculate Vectors of Metrics (Protein-wise)
        pcc_list, mse_list, cos_list = [], [], []
        
        for i in range(yp_mat.shape[1]): # Iterate proteins (columns)
            y_p_col = yp_mat[:, i]
            y_t_col = yt_mat[:, i]
            mask = (~np.isnan(y_p_col)) & (~np.isnan(y_t_col))
            
            if mask.sum() > 2:
                r, _ = pearsonr(y_t_col[mask], y_p_col[mask])
                mse = mean_squared_error(y_t_col[mask], y_p_col[mask])
                denom = (np.linalg.norm(y_p_col[mask]) * np.linalg.norm(y_t_col[mask]))
                cos = np.dot(y_p_col[mask], y_t_col[mask]) / denom if denom > 0 else 0.0
                
                pcc_list.append(r)
                mse_list.append(mse)
                cos_list.append(cos)
            else:
                pcc_list.append(np.nan)
                mse_list.append(np.nan)
                cos_list.append(np.nan)
                
        # Store aggregations
        ds_res[f'{m_name}_Mean_PCC'] = np.nanmean(pcc_list)
        ds_res[f'{m_name}_Mean_MSE'] = np.nanmean(mse_list)
        ds_res[f'{m_name}_Mean_Cosine'] = np.nanmean(cos_list)
        
        # Store raw lists for distribution plotting if needed
        ds_res[f'{m_name}_PCC_List'] = np.array(pcc_list)
        ds_res[f'{m_name}_MSE_List'] = np.array(mse_list)
        ds_res[f'{m_name}_Cosine_List'] = np.array(cos_list)
        
    comparison_results.append(ds_res)

# Create DataFrame
comp_df = pd.DataFrame(comparison_results)
print("\n=== Model Comparison on Intersection ===")
cols = [c for c in comp_df.columns if 'List' not in c]
display(comp_df[cols].round(4))


In [ ]:
# 1. Bar Chart of Means
metrics = ['PCC', 'MSE', 'Cosine']
models = ['Model_Z', 'Model_Rank', 'Model_Bin', 'Model_None']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Comparison: Mean Performance on Intersection Proteins', fontsize=16)

palette = sns.color_palette("muted", len(models))

for i, metric in enumerate(metrics):
    ax = axes[i]
    # Prepare data for plotting
    plot_data = []
    for _, row in comp_df.iterrows():
        ds = row['Dataset']
        for m in models:
            val = row.get(f'{m}_Mean_{metric}', np.nan)
            plot_data.append({'Dataset': ds, 'Model': m, 'Value': val})
            
    df_plot = pd.DataFrame(plot_data)
    sns.barplot(data=df_plot, x='Dataset', y='Value', hue='Model', ax=ax, palette=palette)
    ax.set_title(f'Mean {metric}')
    if i > 0: ax.get_legend().remove()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison_bars.png'))
plt.show()    


In [ ]:
# 2. Winning Model Counts (PCC & MSE)
print('\n=== Winning Model Analysis (Sample-wise Breakdown) ===')

datasets_eval = comp_df['Dataset'].unique()
models_indices = {i: m for i, m in enumerate(models)} # 0: Z, 1: Rank, 2: Bin, 3: None

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Best Performing Model per Protein', fontsize=16)

for j, ds in enumerate(datasets_eval):
    if j >= 3: break # Prevent out of bounds if more than 3 datasets
    # Retrieve row
    row = comp_df[comp_df['Dataset'] == ds].iloc[0]
    
    # --- PCC (Top Row) ---
    ax_pcc = axes[0, j]
    arrays_pcc = [row.get(f'{m}_PCC_List', []) for m in models]
    if not any(len(x) == 0 for x in arrays_pcc):
        stack_pcc = np.vstack(arrays_pcc)
        winners_idx_pcc = np.argmax(stack_pcc, axis=0)
        counts_pcc = Counter(winners_idx_pcc)
        sizes_pcc = [counts_pcc.get(i, 0) for i in range(len(models))]
        ax_pcc.pie(sizes_pcc, labels=[models_indices[i] for i in range(len(models))], autopct='%1.1f%%', colors=palette)
        ax_pcc.set_title(f'PCC Winner - {ds}')
    else:
        ax_pcc.axis('off')
        
    # --- MSE (Bottom Row) ---
    ax_mse = axes[1, j]
    arrays_mse = [row.get(f'{m}_MSE_List', []) for m in models]
    if not any(len(x) == 0 for x in arrays_mse):
        stack_mse = np.vstack(arrays_mse)
        winners_idx_mse = np.argmin(stack_mse, axis=0) # MSE lower is better
        counts_mse = Counter(winners_idx_mse)
        sizes_mse = [counts_mse.get(i, 0) for i in range(len(models))]
        ax_mse.pie(sizes_mse, labels=[models_indices[i] for i in range(len(models))], autopct='%1.1f%%', colors=palette)
        ax_mse.set_title(f'MSE Winner - {ds}')
    else:
        ax_mse.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# High-Performing Proteins Analysis for Model Z
print('\n=== Model_Z High-Performing Proteins Analysis ===')

thresholds = [0.5, 0.6, 0.7]
z_counts = []
z_proteins_dict = {} # To store the actual protein names for text file generation later

for ds in datasets:
    # Find the proteins that were evaluated for this dataset
    PROT_PATH = os.path.join(VAL_DATA_DIR, f"{ds}_proteomics.joblib")
    prot_df = joblib.load(PROT_PATH)
    eval_prots = [p for p in common_proteins if p in prot_df.columns]
    
    # Get the raw lists for Model Z
    ds_row = comp_df[comp_df['Dataset'] == ds].iloc[0]
    pcc_list = ds_row['Model_Z_PCC_List']
    cos_list = ds_row['Model_Z_Cosine_List']
    
    ds_counts = {'Dataset': ds}
    z_proteins_dict[ds] = {}
    
    for t in thresholds:
        # Find indices where both metrics are >= t
        valid_indices = []
        for i, (pcc, cos) in enumerate(zip(pcc_list, cos_list)):
            if not np.isnan(pcc) and not np.isnan(cos) and pcc >= t and cos >= t:
                valid_indices.append(i)
                
        # Get the protein names
        passed_prots = [eval_prots[i] for i in valid_indices]
        
        # Save counts and lists
        ds_counts[f'>= {t}'] = len(passed_prots)
        z_proteins_dict[ds][t] = passed_prots
        
    z_counts.append(ds_counts)

z_counts_df = pd.DataFrame(z_counts)
display(z_counts_df)
